# Day 59 · What a tool call really is

**First-principles question:** Who actually runs the tool: the model or your code?

The full task (learn → build → record) is in this folder's `README.md`. This notebook is your workbench:
run cells top to bottom with **Shift + Enter**, fill in every cell marked **TODO**, and finish with the
**✅ self-check** cell at the end.

In [ ]:
import sys
from pathlib import Path

def find_root():
    """The fde-first-principles folder (it has progress.json)."""
    for p in [Path.cwd(), *Path.cwd().parents]:
        if (p / "progress.json").exists():
            return p
    raise RuntimeError("Open this notebook from inside your fde-first-principles folder.")

ROOT = find_root()
sys.path.insert(0, str(ROOT))  # so `import lab.…` works

import importlib
import json
import pandas as pd
import requests
import lab.acme_tools as acme_tools
importlib.reload(acme_tools)
from lab.acme import load_cases, load_results
from lab.evals import score
from lab.llm import CHAT_MODEL, OLLAMA

cases = load_cases()
by_id = {c["id"]: c for c in cases}

## 1 · The raw JSON of one tool step
No helper, no cache: exactly what goes over the wire.

In [ ]:
request = {
    "model": CHAT_MODEL, "stream": False, "think": False,
    "tools": acme_tools.TOOLS,
    "messages": [{"role": "user", "content": f"Customer id: AC1001\n\nEmail:\n{by_id['c01']['message']}"}],
}
response = requests.post(f"{OLLAMA}/api/chat", json=request, timeout=300).json()
print(json.dumps(response["message"], indent=2, ensure_ascii=False))

**Annotate it:** who wrote the `tool_calls` part? Who will run `lookup_customer`? What would happen if your code never ran it? …

## 2 · Improve the tools
Edit `lab/acme_tools.py` using the article: clearer names and descriptions, examples in descriptions, returning only what the model needs. Re-run the first cell after each edit.

In [ ]:
# TODO: list the case ids that failed because of tools in your day-10 failures.csv (e.g. wrong tool, no lookup)
TOOL_FAILURES = []
print(TOOL_FAILURES)

## 3 · Before and after
Re-run version C on those cases with the improved tools. `run=1` forces fresh answers instead of the cached ones from day 9.

In [ ]:
import lab.agent as agent
importlib.reload(agent)
before = {r["id"]: score(by_id[r["id"]], r)["passed"] for r in load_results("C") if r["id"] in TOOL_FAILURES}
after = {}
for case_id in TOOL_FAILURES:
    answer, trace = agent.run_agent(by_id[case_id], run=1)
    after[case_id] = score(by_id[case_id], answer)["passed"]
pd.DataFrame({"before": before, "after": after})

In [ ]:
# ✅ Self-check: run me last. A clear message tells you what to fix.
assert "message" in response, "Step 1: a raw response"
assert TOOL_FAILURES, "Step 2: list the tool-related failures"
assert set(after) == set(TOOL_FAILURES)
print("All checks passed ✅  Now write 'In my words' in README.md and record your video.")